# 06 — V4 Failure Diagnostics & Unified Development Expansion Plan

**Notebook version:** `SAJU_ML_V4_FAILURE_DIAGNOSTICS_DEV_EXPANSION_20260816`

## tl;dr

This notebook starts only after the V4 discovery tournament has returned:

```text
V4_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE
```

It is a **diagnostic and next-data-design notebook**.

It does **not**:

- fit a new winner;
- search new feature combinations;
- change any V4 label;
- exclude named subjects based on scores;
- open `NEW_CONFIRM`, Validation B, Public CHECK, or Public FINAL;
- create or score an external holdout.

It does:

1. freeze the failed V4 tournament as immutable discovery evidence;
2. decompose failure by axis and collection origin;
3. quantify the collection-origin artifact;
4. inspect ElasticNet feature-selection stability without promoting features;
5. optionally inspect event-mechanism signatures when lineage files are present;
6. create a **single-protocol fresh development collection plan**;
7. output power-planning numbers for a future external holdout, without opening one.

## Context & Methods

### Key assumptions

- Primary evaluation grain remains **subject-macro pairwise**.
- The current 100-pair V4 target is already consumed development data.
- `pair_origin` is **never** a candidate model feature.
- Post-hoc axis/origin/subtype results are descriptive diagnostics only.
- A future development wave must use one collection protocol from the start.
- Chronology balance must emerge from that protocol; do **not** backfill cases merely to force 50/50 after seeing the batch.
- If optional 05 artifacts are missing, the notebook still runs a bounded core diagnostic and explicitly records which diagnostics were unavailable.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_V4_FAILURE_DIAGNOSTICS_DEV_EXPANSION_20260816"
SEED = 20260816
BOOT_N = 10000

# Diagnostic thresholds. These are NOT winner gates.
ORIGIN_ARTIFACT_HIGH = 0.75
MIN_AXIS_SUBJECTS_FOR_INTERPRETATION = 15
MIN_SIGNATURE_SUBJECTS_FOR_INTERPRETATION = 5
ELASTIC_STABLE_SELECTION_FREQ = 0.60
ELASTIC_STABLE_SIGN_CONSISTENCY = 0.70

# Next development collection planning.
CORE_AXES = ["COMPETITIVE", "PROJECT", "STATUS"]
MIN_CORE_AXIS_SUBJECTS = 50
NEXT_DEV_RECOMMENDED_PAIRS = 100
NEXT_DEV_AXIS_ALLOCATION = {
    "COMPETITIVE": 30,
    "PROJECT": 25,
    "STATUS": 45,
}
RECOGNITION_MIN_BEFORE_CORE_USE = 30

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run inside the Chartpalja saju repo; saju_engine.py is required only "
        "as a repo-root marker."
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

REPO_ROOT = find_repo_root()

TARGET_DIR = REPO_ROOT / "research/ml/artifacts/v4_model_ready_targets"
TOURNAMENT_DIR = REPO_ROOT / "research/ml/artifacts/v4_feature_model_tournament"
REVIEW_DIR = REPO_ROOT / "research/ml/artifacts/v4_career_taxonomy_review"
EXPANSION_DIR = REPO_ROOT / "research/ml_corpus/v4_target_expansion"

OUT = REPO_ROOT / "research/ml/artifacts/v4_failure_diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

CORE_PATHS = {
    "pairs": TARGET_DIR / "V4_MODEL_READY_PAIRS.csv",
    "target_decision": TARGET_DIR / "V4_MODEL_READY_TARGET_DECISION.json",
    "tournament_decision": TOURNAMENT_DIR / "V4_TOURNAMENT_DECISION.json",
    "gates": TOURNAMENT_DIR / "V4_candidate_gates.csv",
    "leaderboard": TOURNAMENT_DIR / "V4_nested_leaderboard.csv",
    "bootstrap": TOURNAMENT_DIR / "V4_model_reference_bootstrap.csv",
    "origin": TOURNAMENT_DIR / "V4_origin_robustness.csv",
}

OPTIONAL_PATHS = {
    "axis": TOURNAMENT_DIR / "V4_axis_diagnostics.csv",
    "oof_pair": TOURNAMENT_DIR / "V4_OOF_pair_scores.csv",
    "oof_subject": TOURNAMENT_DIR / "V4_OOF_subject_scores.csv",
    "elastic_by_fold": TOURNAMENT_DIR / "V4_elasticnet_feature_selection_by_fold.csv",
    "elastic_frequency": TOURNAMENT_DIR / "V4_elasticnet_selection_frequency.csv",
    "pair_features": TOURNAMENT_DIR / "V4_pair_diff_feature_table.csv",
    "reviewed_414": REVIEW_DIR / "V4_CAREER_TAXONOMY_SOURCE_REVIEWED_414.csv",
    "expansion_events": EXPANSION_DIR / "V4_EXPANSION_NEW_EVENTS.csv",
}

print("=" * 96)
print("NOTEBOOK:", NOTEBOOK_VERSION)
print("repo:", REPO_ROOT)
print("output:", OUT)
print("=" * 96)

NOTEBOOK: SAJU_ML_V4_FAILURE_DIAGNOSTICS_DEV_EXPANSION_20260816
repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_failure_diagnostics


## Data

### 1. Holdout-safe preflight

In [2]:
missing = [label for label, path in CORE_PATHS.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required V4 artifacts: %s" % missing)

with open(CORE_PATHS["target_decision"], encoding="utf-8") as f:
    target_decision = json.load(f)
with open(CORE_PATHS["tournament_decision"], encoding="utf-8") as f:
    tournament_decision = json.load(f)

assert target_decision["status"] == "V4_TARGET_MODEL_READY"
assert tournament_decision["status"] == "V4_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE"
assert tournament_decision.get("winner") is None

holdout_integrity = tournament_decision.get("holdout_integrity", {})
for key in ["NEW_CONFIRM_loaded", "Validation_B_loaded", "Public_CHECK_loaded", "Public_FINAL_loaded"]:
    assert holdout_integrity.get(key) is False, "Holdout integrity violation: %s" % key

# The notebook never defines paths to sealed sets.
print("V4 failed discovery freeze: PASS")
print("Holdout-safe preflight: PASS")
print("target pairs:", tournament_decision["target"]["n_pairs"])
print("target subjects:", tournament_decision["target"]["n_subjects"])

V4 failed discovery freeze: PASS
Holdout-safe preflight: PASS
target pairs: 100
target subjects: 69


### 2. Load core and optional artifacts

In [3]:
pairs = pd.read_csv(CORE_PATHS["pairs"])
gates = pd.read_csv(CORE_PATHS["gates"])
leaderboard = pd.read_csv(CORE_PATHS["leaderboard"])
bootstrap = pd.read_csv(CORE_PATHS["bootstrap"])
origin = pd.read_csv(CORE_PATHS["origin"])

optional = {}
for label, path in OPTIONAL_PATHS.items():
    if path.exists():
        optional[label] = pd.read_csv(path)
        print("%-20s PRESENT  %s" % (label, path))
    else:
        optional[label] = None
        print("%-20s MISSING  %s" % (label, path))

assert len(pairs) == int(tournament_decision["target"]["n_pairs"])
assert pairs.subject_id.nunique() == int(tournament_decision["target"]["n_subjects"])
assert pairs.source_pair_eligible.astype(bool).all()
assert pairs.model_ready.astype(bool).all()

print("\ncore target:", len(pairs), "pairs /", pairs.subject_id.nunique(), "subjects")

axis                 PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament/V4_axis_diagnostics.csv
oof_pair             PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament/V4_OOF_pair_scores.csv
oof_subject          PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament/V4_OOF_subject_scores.csv
elastic_by_fold      PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament/V4_elasticnet_feature_selection_by_fold.csv
elastic_frequency    PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament/V4_elasticnet_selection_frequency.csv
pair_features        PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament/V4_pair_diff_feature_table.csv
reviewed_414         PRESENT  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4

### 3. Freeze the failed tournament lineage

In [4]:
freeze = {
    "version": "V4_DISCOVERY_FAILURE_FREEZE_V1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_notebook_version": tournament_decision.get("notebook_version"),
    "status": tournament_decision["status"],
    "winner": None,
    "reason": tournament_decision["reason"],
    "hashes": {k: sha256_file(v) for k, v in CORE_PATHS.items()},
    "target": tournament_decision["target"],
    "holdout_integrity": holdout_integrity,
    "rule": (
        "Immutable failed discovery evidence. Do not retune a winner on this 100-pair target. "
        "Diagnostics may guide the design of a NEW development wave only."
    ),
}

with open(OUT / "V4_DISCOVERY_FAILURE_FREEZE.json", "w", encoding="utf-8") as f:
    json.dump(freeze, f, ensure_ascii=False, indent=2)

print(json.dumps(
    {"status": freeze["status"], "winner": freeze["winner"], "rule": freeze["rule"]},
    ensure_ascii=False, indent=2
))

{
  "status": "V4_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE",
  "winner": null,
  "rule": "Immutable failed discovery evidence. Do not retune a winner on this 100-pair target. Diagnostics may guide the design of a NEW development wave only."
}


## Results

### 4. Reproduce the headline failure

In [5]:
winner_eligible = gates.sort_values("overall", ascending=False).copy()

headline_cols = [
    "model", "overall", "p10",
    "delta_vs_nuisance", "p_vs_nuisance",
    "delta_vs_control", "p_vs_control",
    "origin_min", "all_gates",
]
display(winner_eligible[headline_cols])

assert not winner_eligible["all_gates"].astype(bool).any()

best_row = winner_eligible.iloc[0]
print("Best winner-eligible architecture:", best_row["model"])
print("overall:", round(float(best_row["overall"]), 4))
print("p10:", round(float(best_row["p10"]), 4))
print("origin_min:", round(float(best_row["origin_min"]), 4))

,model,overall,p10,delta_vs_nuisance,p_vs_nuisance,delta_vs_control,p_vs_control,origin_min,all_gates
0,ALL_ELASTICNET_AUTO,0.505072,0.289372,0.029710,0.71940,0.028261,0.68185,0.487387,False
1,ALL_L2,0.480580,0.457246,0.005217,0.53045,0.003768,0.52675,0.461982,False
2,TG10_STEM_BRANCH_L2,0.479082,0.444203,0.003720,0.51835,0.002271,0.51415,0.467391,False
3,TG10_STEM_L2,0.453720,0.386957,-0.021643,0.36585,-0.023092,0.37335,0.432609,False
4,TG10_SB_RELATIONS_L2,0.444444,0.392271,-0.030918,0.30450,-0.032367,0.29145,0.414414,False
5,TG10_SB_STRENGTH_L2,0.426135,0.384348,-0.049227,0.21810,-0.050676,0.18700,0.415217,False
6,TG5_GROUP_L2,0.422585,0.391087,-0.052778,0.19850,-0.054227,0.20700,0.411957,False


Best winner-eligible architecture: ALL_ELASTICNET_AUTO
overall: 0.5051
p10: 0.2894
origin_min: 0.4874


### 5. Target structure and collection-origin artifact

In [6]:
target_structure = (
    pairs.groupby(["pair_origin", "axis"])
    .agg(
        n_pairs=("pair_id", "size"),
        n_subjects=("subject_id", "nunique"),
        positive_earlier_share=("positive_earlier_calc", "mean"),
        median_year_gap=("abs_year_gap", "median"),
    )
    .reset_index()
)

axis_structure = (
    pairs.groupby("axis")
    .agg(
        n_pairs=("pair_id", "size"),
        n_subjects=("subject_id", "nunique"),
        positive_earlier_share=("positive_earlier_calc", "mean"),
        median_year_gap=("abs_year_gap", "median"),
    )
    .reset_index()
)

origin_structure = (
    pairs.groupby("pair_origin")
    .agg(
        n_pairs=("pair_id", "size"),
        n_subjects=("subject_id", "nunique"),
        positive_earlier_share=("positive_earlier_calc", "mean"),
        median_year_gap=("abs_year_gap", "median"),
    )
    .reset_index()
)

target_structure.to_csv(OUT / "V4_DIAG_TARGET_ORIGIN_AXIS_STRUCTURE.csv", index=False)
axis_structure.to_csv(OUT / "V4_DIAG_AXIS_STRUCTURE.csv", index=False)
origin_structure.to_csv(OUT / "V4_DIAG_ORIGIN_STRUCTURE.csv", index=False)

display(origin_structure)
display(axis_structure)

chronology_gap = float(
    origin_structure["positive_earlier_share"].max()
    - origin_structure["positive_earlier_share"].min()
)

origin_diag_row = leaderboard[
    leaderboard.model == "NUISANCE_WITH_ORIGIN_DIAGNOSTIC"
]
origin_diag_score = (
    float(origin_diag_row.iloc[0].pairwise_mean)
    if len(origin_diag_row) else np.nan
)

print("origin chronology gap:", round(chronology_gap, 4))
print("forbidden origin diagnostic:", round(origin_diag_score, 4))

,pair_origin,n_pairs,n_subjects,positive_earlier_share,median_year_gap
0,PREEXISTING_414_SOURCE_REVIEW,49,46,0.836735,5.0
1,V4_TARGET_EXPANSION_2026-08-16,51,37,0.000000,3.0


,axis,n_pairs,n_subjects,positive_earlier_share,median_year_gap
0,COMPETITIVE,43,26,0.302326,2.0
1,PROJECT,40,33,0.475000,7.5
2,RECOGNITION,4,3,0.500000,9.5
3,STATUS,13,9,0.538462,5.0


origin chronology gap: 0.8367
forbidden origin diagnostic: 0.8913


### 6. Build repeated-OOF pair averages when available

The richer diagnostics below use the repeated OOF predictions created by 05.

They do **not** refit a model.

In [7]:
pair_avg = None

if optional["oof_pair"] is not None:
    oof_pair = optional["oof_pair"].copy()
    needed = {"model", "pair_id", "subject_id", "pair_origin", "axis", "correct"}
    if not needed.issubset(set(oof_pair.columns)):
        raise RuntimeError("V4_OOF_pair_scores.csv schema mismatch.")

    pair_avg = (
        oof_pair.groupby(["model", "pair_id", "subject_id", "pair_origin", "axis"])["correct"]
        .mean()
        .reset_index()
    )
    print("Repeated-OOF pair averages available:", pair_avg.shape)
else:
    print("Repeated-OOF pair scores unavailable; axis bootstrap/delta diagnostics will be skipped.")

Repeated-OOF pair averages available: (1300, 6)


### 7. Axis diagnostics with subject bootstrap

In [8]:
def subject_macro_bootstrap(frame, n_boot=BOOT_N, seed=SEED):
    subj = frame.groupby("subject_id")["correct"].mean()
    ids = subj.index.to_numpy()
    vals = subj.to_numpy(dtype=float)

    if len(ids) == 0:
        return {
            "n_subjects": 0, "mean": np.nan,
            "ci025": np.nan, "ci975": np.nan, "p_gt_050": np.nan,
        }

    rng = np.random.RandomState(seed)
    draws = []
    for _ in range(n_boot):
        sample_idx = rng.randint(0, len(vals), size=len(vals))
        draws.append(float(vals[sample_idx].mean()))

    arr = np.asarray(draws)
    return {
        "n_subjects": int(len(ids)),
        "mean": float(vals.mean()),
        "ci025": float(np.quantile(arr, 0.025)),
        "ci975": float(np.quantile(arr, 0.975)),
        "p_gt_050": float((arr > 0.50).mean()),
    }

axis_boot_rows = []

if pair_avg is not None:
    for (model, axis), g in pair_avg.groupby(["model", "axis"]):
        stats = subject_macro_bootstrap(
            g,
            n_boot=BOOT_N,
            seed=SEED + sum(ord(x) for x in (str(model) + str(axis))),
        )
        axis_boot_rows.append({
            "model": model,
            "axis": axis,
            "n_pairs": int(len(g)),
            **stats,
            "interpretation_allowed": bool(
                stats["n_subjects"] >= MIN_AXIS_SUBJECTS_FOR_INTERPRETATION
            ),
        })

axis_boot = pd.DataFrame(axis_boot_rows)

if len(axis_boot):
    axis_boot.to_csv(OUT / "V4_DIAG_AXIS_BOOTSTRAP.csv", index=False)
    display(
        axis_boot.sort_values(["axis", "mean"], ascending=[True, False])
        .head(100)
    )
else:
    print("SKIPPED: no OOF pair artifact.")

,model,axis,n_pairs,n_subjects,mean,ci025,ci975,p_gt_050,interpretation_allowed
28,NUISANCE_WITH_ORIGIN_DIAGNOSTIC,COMPETITIVE,43,26,0.807692,0.653846,0.961538,0.9993,True
0,AGE_LATER_FIXED,COMPETITIVE,43,26,0.625000,0.448718,0.791667,0.9124,True
8,ALL_ELASTICNET_AUTO,COMPETITIVE,43,26,0.554487,0.469872,0.643590,0.8871,True
12,ALL_L2,COMPETITIVE,43,26,0.551026,0.402564,0.696923,0.7457,True
20,HGB_ALL_EXPLORATORY,COMPETITIVE,43,26,0.481026,0.370513,0.587692,0.3657,True
44,TG10_STEM_L2,COMPETITIVE,43,26,0.472051,0.336147,0.609231,0.3413,True
40,TG10_STEM_BRANCH_L2,COMPETITIVE,43,26,0.458590,0.330125,0.591032,0.2667,True
36,TG10_SB_STRENGTH_L2,COMPETITIVE,43,26,0.438590,0.308590,0.572564,0.1836,True
48,TG5_GROUP_L2,COMPETITIVE,43,26,0.433013,0.301603,0.564423,0.1557,True
32,TG10_SB_RELATIONS_L2,COMPETITIVE,43,26,0.419231,0.282051,0.556410,0.1234,True


### 8. Axis-specific incremental value vs nuisance and Control

In [9]:
def bootstrap_subject_delta(model_frame, ref_frame, n_boot=BOOT_N, seed=SEED):
    m = model_frame.groupby("subject_id")["correct"].mean()
    r = ref_frame.groupby("subject_id")["correct"].mean()
    shared = pd.concat([m.rename("model"), r.rename("reference")], axis=1).dropna()

    if len(shared) == 0:
        return {
            "n_subjects": 0, "observed_delta": np.nan,
            "ci025": np.nan, "ci975": np.nan, "p_delta_gt_0": np.nan,
        }

    deltas = (shared["model"] - shared["reference"]).to_numpy(dtype=float)
    rng = np.random.RandomState(seed)
    draws = []

    for _ in range(n_boot):
        idx = rng.randint(0, len(deltas), size=len(deltas))
        draws.append(float(deltas[idx].mean()))

    arr = np.asarray(draws)
    return {
        "n_subjects": int(len(shared)),
        "observed_delta": float(deltas.mean()),
        "ci025": float(np.quantile(arr, 0.025)),
        "ci975": float(np.quantile(arr, 0.975)),
        "p_delta_gt_0": float((arr > 0).mean()),
    }

ASTRO_MODELS = gates["model"].tolist()
REFERENCES = ["NUISANCE_AGE_AXIS", "CONTROL_FIXED"]

axis_delta_rows = []

if pair_avg is not None:
    for axis in sorted(pair_avg.axis.unique()):
        axis_data = pair_avg[pair_avg.axis == axis]

        for model in ASTRO_MODELS:
            mf = axis_data[axis_data.model == model]
            if mf.empty:
                continue

            for reference in REFERENCES:
                rf = axis_data[axis_data.model == reference]
                if rf.empty:
                    continue

                stats = bootstrap_subject_delta(
                    mf, rf,
                    n_boot=BOOT_N,
                    seed=SEED + sum(ord(x) for x in (axis + model + reference)),
                )
                axis_delta_rows.append({
                    "axis": axis,
                    "model": model,
                    "reference": reference,
                    **stats,
                    "interpretation_allowed": bool(
                        stats["n_subjects"] >= MIN_AXIS_SUBJECTS_FOR_INTERPRETATION
                    ),
                })

axis_delta = pd.DataFrame(axis_delta_rows)

if len(axis_delta):
    axis_delta.to_csv(OUT / "V4_DIAG_AXIS_REFERENCE_DELTA.csv", index=False)
    display(
        axis_delta.sort_values(
            ["axis", "reference", "observed_delta"],
            ascending=[True, True, False],
        ).head(120)
    )
else:
    print("SKIPPED: no OOF pair artifact.")

,axis,model,reference,n_subjects,observed_delta,ci025,ci975,p_delta_gt_0,interpretation_allowed
1,COMPETITIVE,ALL_ELASTICNET_AUTO,CONTROL_FIXED,26,0.173718,-0.019231,0.364760,0.9595,True
3,COMPETITIVE,ALL_L2,CONTROL_FIXED,26,0.170256,-0.043846,0.373077,0.9472,True
7,COMPETITIVE,TG10_STEM_L2,CONTROL_FIXED,26,0.091282,-0.135647,0.301282,0.7952,True
5,COMPETITIVE,TG10_STEM_BRANCH_L2,CONTROL_FIXED,26,0.077821,-0.109615,0.255651,0.7978,True
11,COMPETITIVE,TG10_SB_STRENGTH_L2,CONTROL_FIXED,26,0.057821,-0.081410,0.200769,0.7928,True
13,COMPETITIVE,TG5_GROUP_L2,CONTROL_FIXED,26,0.052244,-0.163486,0.260577,0.6848,True
9,COMPETITIVE,TG10_SB_RELATIONS_L2,CONTROL_FIXED,26,0.038462,-0.157692,0.228205,0.6513,True
0,COMPETITIVE,ALL_ELASTICNET_AUTO,NUISANCE_AGE_AXIS,26,0.158333,0.035256,0.290385,0.9954,True
2,COMPETITIVE,ALL_L2,NUISANCE_AGE_AXIS,26,0.154872,-0.035904,0.352308,0.9441,True
6,COMPETITIVE,TG10_STEM_L2,NUISANCE_AGE_AXIS,26,0.075897,-0.102314,0.262051,0.7880,True


### 9. Origin robustness and axis × origin diagnostics

In [10]:
origin_pivot = origin.pivot(
    index="model",
    columns="pair_origin",
    values="subject_macro_pairwise",
)

origin_gap_rows = []
for model, row in origin_pivot.iterrows():
    vals = row.dropna().to_numpy(dtype=float)
    if len(vals) < 2:
        continue
    origin_gap_rows.append({
        "model": model,
        "origin_min": float(vals.min()),
        "origin_max": float(vals.max()),
        "origin_gap": float(vals.max() - vals.min()),
    })

origin_gap_df = pd.DataFrame(origin_gap_rows).sort_values(
    ["origin_min", "origin_gap"], ascending=[False, True]
)
origin_gap_df.to_csv(OUT / "V4_DIAG_ORIGIN_GAP.csv", index=False)
display(origin_gap_df)

axis_origin_rows = []
if pair_avg is not None:
    for (model, axis, pair_origin), g in pair_avg.groupby(
        ["model", "axis", "pair_origin"]
    ):
        subj = g.groupby("subject_id")["correct"].mean()
        axis_origin_rows.append({
            "model": model,
            "axis": axis,
            "pair_origin": pair_origin,
            "n_pairs": int(len(g)),
            "n_subjects": int(g.subject_id.nunique()),
            "subject_macro_pairwise": float(subj.mean()),
        })

axis_origin = pd.DataFrame(axis_origin_rows)
if len(axis_origin):
    axis_origin.to_csv(OUT / "V4_DIAG_AXIS_ORIGIN_MATRIX.csv", index=False)
    display(axis_origin.head(120))

,model,origin_min,origin_max,origin_gap
7,NUISANCE_WITH_ORIGIN_DIAGNOSTIC,0.836957,1.000000,0.163043
2,ALL_ELASTICNET_AUTO,0.487387,0.508696,0.021308
10,TG10_STEM_BRANCH_L2,0.467391,0.470450,0.003059
3,ALL_L2,0.461982,0.478261,0.016279
11,TG10_STEM_L2,0.432609,0.483964,0.051355
5,HGB_ALL_EXPLORATORY,0.421802,0.436957,0.015155
9,TG10_SB_STRENGTH_L2,0.415217,0.428468,0.013251
8,TG10_SB_RELATIONS_L2,0.414414,0.460870,0.046455
12,TG5_GROUP_L2,0.411957,0.449550,0.037593
4,CONTROL_FIXED,0.407207,0.521739,0.114532


,model,axis,pair_origin,n_pairs,n_subjects,subject_macro_pairwise
0,AGE_LATER_FIXED,COMPETITIVE,PREEXISTING_414_SOURCE_REVIEW,18,17,0.294118
1,AGE_LATER_FIXED,COMPETITIVE,V4_TARGET_EXPANSION_2026-08-16,25,13,1.000000
2,AGE_LATER_FIXED,PROJECT,PREEXISTING_414_SOURCE_REVIEW,20,19,0.052632
3,AGE_LATER_FIXED,PROJECT,V4_TARGET_EXPANSION_2026-08-16,20,20,1.000000
4,AGE_LATER_FIXED,RECOGNITION,PREEXISTING_414_SOURCE_REVIEW,4,3,0.500000
...,...,...,...,...,...,...
86,TG5_GROUP_L2,PROJECT,PREEXISTING_414_SOURCE_REVIEW,20,19,0.347368
87,TG5_GROUP_L2,PROJECT,V4_TARGET_EXPANSION_2026-08-16,20,20,0.500000
88,TG5_GROUP_L2,RECOGNITION,PREEXISTING_414_SOURCE_REVIEW,4,3,0.250000
89,TG5_GROUP_L2,STATUS,PREEXISTING_414_SOURCE_REVIEW,7,7,0.585714


### 10. Event-mechanism signature diagnostics

This section is lineage-aware:

- preexisting 414-review events use `final_subtype` when available;
- expansion events do not share that legacy subtype vocabulary, so their
  predeclared `measurement_basis` is used as a **mechanism signature**, not
  silently relabeled as a traditional subtype.

Small cells are never promoted into conclusions.

In [11]:
event_meta = {}

reviewed = optional["reviewed_414"]
if reviewed is not None:
    for _, r in reviewed.iterrows():
        event_meta[str(r["event_id"])] = {
            "mechanism": str(r.get("final_subtype", "UNKNOWN")),
            "mechanism_source": "final_subtype",
        }

exp_events = optional["expansion_events"]
if exp_events is not None:
    for _, r in exp_events.iterrows():
        event_meta[str(r["event_id"])] = {
            "mechanism": str(r.get("measurement_basis", "UNKNOWN")),
            "mechanism_source": "measurement_basis",
        }

pairs_sig = pairs.copy()

def mechanism_for(event_id):
    meta = event_meta.get(str(event_id))
    return meta["mechanism"] if meta else "UNAVAILABLE"

def mechanism_source_for(event_id):
    meta = event_meta.get(str(event_id))
    return meta["mechanism_source"] if meta else "UNAVAILABLE"

pairs_sig["positive_mechanism"] = pairs_sig["positive_event_id"].map(mechanism_for)
pairs_sig["negative_mechanism"] = pairs_sig["negative_event_id"].map(mechanism_for)
pairs_sig["positive_mechanism_source"] = pairs_sig["positive_event_id"].map(
    mechanism_source_for
)
pairs_sig["negative_mechanism_source"] = pairs_sig["negative_event_id"].map(
    mechanism_source_for
)
pairs_sig["mechanism_signature"] = (
    pairs_sig["axis"].astype(str)
    + "::"
    + pairs_sig["positive_mechanism"].astype(str)
    + "__VS__"
    + pairs_sig["negative_mechanism"].astype(str)
)

signature_counts = (
    pairs_sig.groupby("mechanism_signature")
    .agg(
        n_pairs=("pair_id", "size"),
        n_subjects=("subject_id", "nunique"),
        positive_earlier_share=("positive_earlier_calc", "mean"),
    )
    .reset_index()
    .sort_values(["n_subjects", "n_pairs"], ascending=False)
)
signature_counts["interpretation_allowed"] = (
    signature_counts["n_subjects"] >= MIN_SIGNATURE_SUBJECTS_FOR_INTERPRETATION
)
signature_counts.to_csv(OUT / "V4_DIAG_MECHANISM_SIGNATURE_COUNTS.csv", index=False)

display(signature_counts.head(50))

signature_perf_rows = []
if pair_avg is not None and len(event_meta):
    pair_sig_map = pairs_sig[
        ["pair_id", "mechanism_signature"]
    ].drop_duplicates()

    enriched = pair_avg.merge(pair_sig_map, on="pair_id", how="left")

    for (model, signature), g in enriched.groupby(
        ["model", "mechanism_signature"]
    ):
        subj = g.groupby("subject_id")["correct"].mean()
        signature_perf_rows.append({
            "model": model,
            "mechanism_signature": signature,
            "n_pairs": int(len(g)),
            "n_subjects": int(g.subject_id.nunique()),
            "subject_macro_pairwise": float(subj.mean()),
            "interpretation_allowed": bool(
                g.subject_id.nunique() >= MIN_SIGNATURE_SUBJECTS_FOR_INTERPRETATION
            ),
        })

signature_perf = pd.DataFrame(signature_perf_rows)
if len(signature_perf):
    signature_perf.to_csv(
        OUT / "V4_DIAG_MECHANISM_SIGNATURE_PERFORMANCE.csv",
        index=False,
    )
    display(
        signature_perf[
            signature_perf.interpretation_allowed
        ].sort_values(
            ["mechanism_signature", "subject_macro_pairwise"],
            ascending=[True, False],
        ).head(100)
    )
else:
    print("Mechanism performance skipped: lineage or OOF artifact unavailable.")

,mechanism_signature,n_pairs,n_subjects,positive_earlier_share,interpretation_allowed
1,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.687500,True
2,COMPETITIVE::explicit_outcome__VS__explicit_ou...,25,13,0.000000,True
7,PROJECT::CREATIVE_OR_MEDIA_PROJECT__VS__CREATI...,12,11,0.916667,True
9,PROJECT::commercial_success__VS__commercial_fa...,5,5,0.000000,True
19,RECOGNITION::FORMAL_RECOGNITION__VS__FORMAL_RE...,4,3,0.500000,False
13,PROJECT::commercial_success__VS__overall_proje...,3,3,0.000000,False
18,PROJECT::project_success__VS__project_cancella...,3,3,0.000000,False
20,STATUS::APPOINTMENT_GAIN__VS__ELECTION_LOSS,3,3,1.000000,False
3,PROJECT::CREATIVE_OR_MEDIA_PROJECT__VS__CANCEL...,2,2,1.000000,False
5,PROJECT::CREATIVE_OR_MEDIA_PROJECT__VS__COMMER...,2,2,1.000000,False


,model,mechanism_signature,n_pairs,n_subjects,subject_macro_pairwise,interpretation_allowed
30,AGE_YOUNGER_FIXED,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.687500,True
204,NUISANCE_WITH_ORIGIN_DIAGNOSTIC,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.687500,True
88,ALL_L2,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.625000,True
291,TG10_STEM_BRANCH_L2,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.562500,True
59,ALL_ELASTICNET_AUTO,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.537500,True
320,TG10_STEM_L2,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.525000,True
233,TG10_SB_RELATIONS_L2,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.512500,True
262,TG10_SB_STRENGTH_L2,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.500000,True
146,HGB_ALL_EXPLORATORY,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.487500,True
175,NUISANCE_AGE_AXIS,COMPETITIVE::TITLE_WIN__VS__MAJOR_LOSS,16,16,0.425000,True


### 11. ElasticNet feature-selection stability

In [12]:
elastic_fold = optional["elastic_by_fold"]

if elastic_fold is not None:
    ef = elastic_fold.copy()
    required = {"feature", "selected", "coefficient"}
    if not required.issubset(set(ef.columns)):
        raise RuntimeError("ElasticNet fold-selection artifact schema mismatch.")

    def feature_family(feature):
        x = str(feature)
        x = x.replace("diff__", "", 1)
        if x.startswith("tg10__"):
            return "TG10"
        if x.startswith("tengod__"):
            return "TG5"
        if x.startswith("tgxstrength__"):
            return "TGX_STRENGTH"
        if x.startswith("relation__"):
            return "RELATION"
        return x.split("__", 1)[0].upper()

    stability_rows = []
    for feature, g in ef.groupby("feature"):
        coefs = pd.to_numeric(g["coefficient"], errors="coerce").fillna(0.0).to_numpy()
        selected = np.abs(coefs) > 1e-8
        nz = coefs[selected]

        if len(nz):
            pos_share = float((nz > 0).mean())
            sign_consistency = max(pos_share, 1.0 - pos_share)
        else:
            pos_share = np.nan
            sign_consistency = np.nan

        stability_rows.append({
            "feature": feature,
            "family": feature_family(feature),
            "n_folds": int(len(g)),
            "selection_frequency": float(selected.mean()),
            "mean_coefficient": float(coefs.mean()),
            "mean_abs_coefficient": float(np.abs(coefs).mean()),
            "positive_share_when_selected": pos_share,
            "sign_consistency_when_selected": sign_consistency,
        })

    feature_stability = pd.DataFrame(stability_rows)
    feature_stability["stable_diagnostic_only"] = (
        (feature_stability["selection_frequency"] >= ELASTIC_STABLE_SELECTION_FREQ)
        & (
            feature_stability["sign_consistency_when_selected"]
            >= ELASTIC_STABLE_SIGN_CONSISTENCY
        )
    )
    feature_stability = feature_stability.sort_values(
        ["selection_frequency", "mean_abs_coefficient"],
        ascending=False,
    )
    feature_stability.to_csv(
        OUT / "V4_DIAG_ELASTICNET_FEATURE_STABILITY.csv",
        index=False,
    )

    family_stability = (
        feature_stability.groupby("family")
        .agg(
            n_features=("feature", "size"),
            mean_selection_frequency=("selection_frequency", "mean"),
            max_selection_frequency=("selection_frequency", "max"),
            n_stable_diagnostic_only=("stable_diagnostic_only", "sum"),
        )
        .reset_index()
        .sort_values(
            ["n_stable_diagnostic_only", "max_selection_frequency"],
            ascending=False,
        )
    )
    family_stability.to_csv(
        OUT / "V4_DIAG_FEATURE_FAMILY_STABILITY.csv",
        index=False,
    )

    display(feature_stability.head(50))
    display(family_stability)
else:
    feature_stability = pd.DataFrame()
    family_stability = pd.DataFrame()
    print("SKIPPED: V4_elasticnet_feature_selection_by_fold.csv not found.")

,feature,family,n_folds,selection_frequency,mean_coefficient,mean_abs_coefficient,positive_share_when_selected,sign_consistency_when_selected,stable_diagnostic_only
200,diff__relation__sw_dw_branch_punish,RELATION,25,0.40,0.095066,0.095066,1.0,1.0,False
233,diff__shinsal__sw_세운_신살_길신__문창귀인_,SHINSAL,25,0.40,-0.087537,0.087537,0.0,1.0,False
342,diff__tgxstrength__sw_qi_sha_weak,TGX_STRENGTH,25,0.40,-0.082125,0.082125,0.0,1.0,False
201,diff__relation__sw_dw_stem_clash,RELATION,25,0.40,-0.071559,0.071559,0.0,1.0,False
369,diff__unseong__dw_state_절,UNSEONG,25,0.40,0.040334,0.040334,1.0,1.0,False
338,diff__tgxstrength__sw_pian_yin_strong,TGX_STRENGTH,25,0.36,0.112162,0.112162,1.0,1.0,False
266,diff__tg10__dw_branch_hidden_zheng_guan,TG10,25,0.36,-0.108397,0.108397,0.0,1.0,False
231,diff__shinsal__sw_세운_신살_길신__count,SHINSAL,25,0.36,0.095876,0.095876,1.0,1.0,False
401,diff__yongshin__sw_yong_stem_fit,YONGSHIN,25,0.36,-0.088503,0.088503,0.0,1.0,False
199,diff__relation__sw_dw_branch_harm,RELATION,25,0.36,0.085015,0.085015,1.0,1.0,False


,family,n_features,mean_selection_frequency,max_selection_frequency,n_stable_diagnostic_only
3,RELATION,21,0.131429,0.40,0
4,SHINSAL,43,0.098605,0.40,0
7,TGX_STRENGTH,60,0.060667,0.40,0
8,UNSEONG,28,0.120000,0.40,0
5,TG10,40,0.087000,0.36,0
9,YONGSHIN,16,0.100000,0.36,0
0,BASIC,166,0.035904,0.32,0
6,TG5,10,0.020000,0.08,0
1,ENGINE,8,0.000000,0.00,0
2,HIDDEN,10,0.000000,0.00,0


### 12. Failure taxonomy

In [13]:
failure_rows = []

def add_failure(code, severity, evidence, implication):
    failure_rows.append({
        "code": code,
        "severity": severity,
        "evidence": evidence,
        "implication": implication,
    })

best_model = str(best_row["model"])
best_overall = float(best_row["overall"])
best_p10 = float(best_row["p10"])
best_origin_min = float(best_row["origin_min"])
best_p_n = float(best_row["p_vs_nuisance"])
best_p_c = float(best_row["p_vs_control"])

if origin_diag_score >= ORIGIN_ARTIFACT_HIGH and chronology_gap >= 0.50:
    add_failure(
        "COLLECTION_ORIGIN_CONFOUNDING_HIGH",
        "HIGH",
        "forbidden origin diagnostic=%.3f; origin chronology gap=%.3f"
        % (origin_diag_score, chronology_gap),
        "Current 100-pair target is consumed diagnostic development, not a clean external-like corpus.",
    )

if best_overall < 0.55:
    add_failure(
        "ASTRO_SIGNAL_WEAK_OVERALL",
        "HIGH",
        "%s overall=%.3f" % (best_model, best_overall),
        "No current astrology architecture is strong enough to freeze.",
    )

if best_p10 < 0.48:
    add_failure(
        "BEST_MODEL_LOWER_TAIL_UNSTABLE",
        "HIGH",
        "%s p10=%.3f" % (best_model, best_p10),
        "Repeated subject-CV performance is highly unstable.",
    )

if best_origin_min < 0.50:
    add_failure(
        "BEST_MODEL_ORIGIN_REPLICATION_FAIL",
        "HIGH",
        "%s origin_min=%.3f" % (best_model, best_origin_min),
        "Best model does not independently clear chance in both collection origins.",
    )

if best_p_n < 0.80 or best_p_c < 0.80:
    add_failure(
        "INCREMENTAL_VALUE_NOT_SECURE",
        "HIGH",
        "P(delta>0) nuisance=%.3f; Control=%.3f" % (best_p_n, best_p_c),
        "Observed positive deltas are not statistically secure enough for promotion.",
    )

lead = leaderboard.set_index("model")
if "TG10_STEM_BRANCH_L2" in lead.index and "TG5_GROUP_L2" in lead.index:
    recovery = float(
        lead.loc["TG10_STEM_BRANCH_L2", "pairwise_mean"]
        - lead.loc["TG5_GROUP_L2", "pairwise_mean"]
    )
    if recovery > 0.03 and float(
        lead.loc["TG10_STEM_BRANCH_L2", "pairwise_mean"]
    ) < 0.50:
        add_failure(
            "EXACT_TENGOD_REPRESENTATION_RECOVERY_ONLY",
            "MEDIUM",
            "TG10 stem+branch improves over TG5 by %.3f but remains <0.50" % recovery,
            "Richer Ten-God encoding is worth retaining as a next-dev candidate, not a winner.",
        )

for _, r in axis_structure.iterrows():
    if int(r.n_subjects) < MIN_AXIS_SUBJECTS_FOR_INTERPRETATION:
        add_failure(
            "AXIS_SPARSE_%s" % str(r.axis),
            "MEDIUM",
            "%s has %d subjects / %d pairs"
            % (r.axis, int(r.n_subjects), int(r.n_pairs)),
            "Do not make axis-specific claims from this cell.",
        )

if len(feature_stability):
    n_stable = int(feature_stability["stable_diagnostic_only"].sum())
    if n_stable == 0:
        add_failure(
            "ELASTICNET_FEATURE_SELECTION_UNSTABLE",
            "MEDIUM",
            "No feature meets >=%.0f%% selection and >=%.0f%% selected-sign consistency."
            % (
                100 * ELASTIC_STABLE_SELECTION_FREQ,
                100 * ELASTIC_STABLE_SIGN_CONSISTENCY,
            ),
            "Do not freeze individual features or coefficients from this discovery run.",
        )

failure_taxonomy = pd.DataFrame(failure_rows)
failure_taxonomy.to_csv(OUT / "V4_FAILURE_TAXONOMY.csv", index=False)
display(failure_taxonomy)

,code,severity,evidence,implication
0,COLLECTION_ORIGIN_CONFOUNDING_HIGH,HIGH,forbidden origin diagnostic=0.891; origin chro...,Current 100-pair target is consumed diagnostic...
1,ASTRO_SIGNAL_WEAK_OVERALL,HIGH,ALL_ELASTICNET_AUTO overall=0.505,No current astrology architecture is strong en...
2,BEST_MODEL_LOWER_TAIL_UNSTABLE,HIGH,ALL_ELASTICNET_AUTO p10=0.289,Repeated subject-CV performance is highly unst...
3,BEST_MODEL_ORIGIN_REPLICATION_FAIL,HIGH,ALL_ELASTICNET_AUTO origin_min=0.487,Best model does not independently clear chance...
4,INCREMENTAL_VALUE_NOT_SECURE,HIGH,P(delta>0) nuisance=0.719; Control=0.682,Observed positive deltas are not statistically...
5,EXACT_TENGOD_REPRESENTATION_RECOVERY_ONLY,MEDIUM,TG10 stem+branch improves over TG5 by 0.056 bu...,Richer Ten-God encoding is worth retaining as ...
6,AXIS_SPARSE_RECOGNITION,MEDIUM,RECOGNITION has 3 subjects / 4 pairs,Do not make axis-specific claims from this cell.
7,AXIS_SPARSE_STATUS,MEDIUM,STATUS has 9 subjects / 13 pairs,Do not make axis-specific claims from this cell.
8,ELASTICNET_FEATURE_SELECTION_UNSTABLE,MEDIUM,No feature meets >=60% selection and >=70% sel...,Do not freeze individual features or coefficie...


### 13. Next development collection plan

The next wave is designed to remove the biggest current weakness: **two opposing collection cohorts**.

#### Single protocol

For every new subject:

1. choose the subject universe **before** any astrology score is generated;
2. require the predeclared birth-time/source quality;
3. assign one primary Career axis before searching for outcomes;
4. enumerate source-verifiable positive and negative events on that same axis;
5. choose the opposite-polarity pair using one deterministic rule:
   **minimum absolute year gap**; deterministic source/date tie-break if necessary;
6. do not choose cases because positive happened earlier or later;
7. freeze all pairs and sources before astrology feature extraction;
8. only after freeze, measure chronology balance;
9. if chronology is still imbalanced, collect another complete wave under the same protocol;
   **do not directional-backfill** cases to force 50/50.

The current 100 pairs remain consumed diagnostics.

In [14]:
current_axis = (
    pairs.groupby("axis")
    .agg(
        current_pairs=("pair_id", "size"),
        current_subjects=("subject_id", "nunique"),
    )
    .reset_index()
)

plan_rows = []
for axis in CORE_AXES:
    row = current_axis[current_axis.axis == axis]
    current_subjects = int(row.iloc[0].current_subjects) if len(row) else 0
    current_pairs = int(row.iloc[0].current_pairs) if len(row) else 0

    plan_rows.append({
        "axis": axis,
        "role": "CORE_NEXT_DEV",
        "current_subjects": current_subjects,
        "current_pairs": current_pairs,
        "minimum_subject_deficit_to_50": max(0, MIN_CORE_AXIS_SUBJECTS - current_subjects),
        "recommended_fresh_pairs_next_wave": NEXT_DEV_AXIS_ALLOCATION[axis],
        "expected_subjects_after_one_pair_per_new_subject": (
            current_subjects + NEXT_DEV_AXIS_ALLOCATION[axis]
        ),
    })

recognition_row = current_axis[current_axis.axis == "RECOGNITION"]
rec_subjects = int(recognition_row.iloc[0].current_subjects) if len(recognition_row) else 0
rec_pairs = int(recognition_row.iloc[0].current_pairs) if len(recognition_row) else 0

plan_rows.append({
    "axis": "RECOGNITION",
    "role": "EXPLORATORY_DEFER_FROM_CORE",
    "current_subjects": rec_subjects,
    "current_pairs": rec_pairs,
    "minimum_subject_deficit_to_50": np.nan,
    "recommended_fresh_pairs_next_wave": 0,
    "expected_subjects_after_one_pair_per_new_subject": rec_subjects,
})

dev_plan = pd.DataFrame(plan_rows)
dev_plan.to_csv(OUT / "V4_NEXT_DEV_COLLECTION_PLAN.csv", index=False)
display(dev_plan)

core_minimum_deficit = int(
    dev_plan[
        dev_plan.role == "CORE_NEXT_DEV"
    ]["minimum_subject_deficit_to_50"].sum()
)

print("Minimum core subject-axis deficit to 50 each:", core_minimum_deficit)
print("Recommended fresh unified next-dev wave:", NEXT_DEV_RECOMMENDED_PAIRS)
print("Allocation:", NEXT_DEV_AXIS_ALLOCATION)
print(
    "Recognition remains exploratory until at least",
    RECOGNITION_MIN_BEFORE_CORE_USE,
    "independent subjects."
)

,axis,role,current_subjects,current_pairs,minimum_subject_deficit_to_50,recommended_fresh_pairs_next_wave,expected_subjects_after_one_pair_per_new_subject
0,COMPETITIVE,CORE_NEXT_DEV,26,43,24.0,30,56
1,PROJECT,CORE_NEXT_DEV,33,40,17.0,25,58
2,STATUS,CORE_NEXT_DEV,9,13,41.0,45,54
3,RECOGNITION,EXPLORATORY_DEFER_FROM_CORE,3,4,NaN,0,3


Minimum core subject-axis deficit to 50 each: 82
Recommended fresh unified next-dev wave: 100
Allocation: {'COMPETITIVE': 30, 'PROJECT': 25, 'STATUS': 45}
Recognition remains exploratory until at least 30 independent subjects.


### 14. Future holdout power planning — planning only

In [15]:
# Approximate one-sided alpha=.05, power=.80 sample size for a Bernoulli
# subject-level pairwise endpoint under a simple normal approximation.
# This is planning guidance, not a claim that the current data satisfy
# independent Bernoulli assumptions.
Z_ALPHA_ONE_SIDED = 1.6448536269514722
Z_POWER_80 = 0.8416212335729143

power_rows = []
for true_pairwise in [0.55, 0.56, 0.57, 0.58, 0.60]:
    numerator = (
        Z_ALPHA_ONE_SIDED * math.sqrt(0.25)
        + Z_POWER_80 * math.sqrt(true_pairwise * (1 - true_pairwise))
    ) ** 2
    denominator = (true_pairwise - 0.50) ** 2
    n = int(math.ceil(numerator / denominator))

    power_rows.append({
        "assumed_true_subject_macro_pairwise": true_pairwise,
        "approx_subjects_for_one_sided_5pct_80pct_power": n,
        "note": "Planning approximation; final holdout size must be preregistered from the fresh-dev effect and tie structure.",
    })

power_plan = pd.DataFrame(power_rows)
power_plan.to_csv(OUT / "V4_FUTURE_HOLDOUT_POWER_PLANNING.csv", index=False)
display(power_plan)

,assumed_true_subject_macro_pairwise,approx_subjects_for_one_sided_5pct_80pct_power,note
0,0.55,617,Planning approximation; final holdout size mus...
1,0.56,428,Planning approximation; final holdout size mus...
2,0.57,314,Planning approximation; final holdout size mus...
3,0.58,240,Planning approximation; final holdout size mus...
4,0.60,153,Planning approximation; final holdout size mus...


### 15. Diagnostic decision and next rule

In [16]:
optional_presence = {
    label: bool(df is not None)
    for label, df in optional.items()
}

core_axis_counts = {
    str(r.axis): {
        "pairs": int(r.n_pairs),
        "subjects": int(r.n_subjects),
    }
    for _, r in axis_structure.iterrows()
}

shortlist = [
    "TG10_STEM_BRANCH_L2",
    "ALL_L2",
    "ALL_ELASTICNET_AUTO",
]

decision = {
    "version": "SAJU_ML_V4_FAILURE_DIAGNOSTIC_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": "V4_FAIL_DIAGNOSED_COLLECT_UNIFIED_DEV",
    "winner": None,
    "source_tournament_status": tournament_decision["status"],
    "current_100_pair_role": "CONSUMED_DIAGNOSTIC_DEVELOPMENT",
    "key_evidence": {
        "best_winner_eligible_model": best_model,
        "best_overall": best_overall,
        "best_p10": best_p10,
        "best_origin_min": best_origin_min,
        "origin_chronology_gap": chronology_gap,
        "forbidden_origin_diagnostic": origin_diag_score,
        "core_axis_counts": core_axis_counts,
    },
    "failure_taxonomy": failure_taxonomy.to_dict(orient="records"),
    "next_development": {
        "protocol": "SINGLE_UNIFIED_COLLECTION_PROTOCOL",
        "recommended_fresh_pairs": NEXT_DEV_RECOMMENDED_PAIRS,
        "axis_allocation": NEXT_DEV_AXIS_ALLOCATION,
        "minimum_core_axis_subjects_for_diagnostic_interpretation": MIN_CORE_AXIS_SUBJECTS,
        "recognition_core_deferred_until_subjects": RECOGNITION_MIN_BEFORE_CORE_USE,
        "chronology_rule": (
            "Do not directional-backfill. Freeze a complete wave first; "
            "if chronology remains imbalanced, collect another complete wave under the same protocol."
        ),
        "pair_selection_rule": (
            "Within the preassigned axis, enumerate verified opposite-polarity events "
            "and choose the minimum-absolute-year-gap pair using a deterministic tie-break."
        ),
    },
    "next_dev_architecture_shortlist_diagnostic_only": shortlist,
    "shortlist_note": (
        "This is not a winner freeze. It only limits the next development comparison "
        "to previously tested families; no new feature search is authorized here."
    ),
    "optional_artifacts_present": optional_presence,
    "holdout_integrity": {
        "NEW_CONFIRM_loaded": False,
        "Validation_B_loaded": False,
        "Public_CHECK_loaded": False,
        "Public_FINAL_loaded": False,
    },
    "next_rule": (
        "Collect the fresh unified development wave BEFORE generating astrology scores. "
        "Then run a preregistered small-family development comparison. "
        "Do not construct or open a new external holdout until a candidate passes discovery gates."
    ),
}

with open(OUT / "V4_FAILURE_DIAGNOSTIC_DECISION.json", "w", encoding="utf-8") as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)

with open(OUT / "V4_FAILURE_DIAGNOSTIC_SUMMARY.json", "w", encoding="utf-8") as f:
    json.dump({
        "status": decision["status"],
        "winner": None,
        "key_evidence": decision["key_evidence"],
        "next_development": decision["next_development"],
        "next_rule": decision["next_rule"],
    }, f, ensure_ascii=False, indent=2)

print(json.dumps({
    "status": decision["status"],
    "winner": decision["winner"],
    "recommended_fresh_pairs": NEXT_DEV_RECOMMENDED_PAIRS,
    "axis_allocation": NEXT_DEV_AXIS_ALLOCATION,
    "next_rule": decision["next_rule"],
}, ensure_ascii=False, indent=2))

{
  "status": "V4_FAIL_DIAGNOSED_COLLECT_UNIFIED_DEV",
  "winner": null,
  "recommended_fresh_pairs": 100,
  "axis_allocation": {
    "COMPETITIVE": 30,
    "PROJECT": 25,
    "STATUS": 45
  },
  "next_rule": "Collect the fresh unified development wave BEFORE generating astrology scores. Then run a preregistered small-family development comparison. Do not construct or open a new external holdout until a candidate passes discovery gates."
}


## Takeaways

After **Kernel Restart → Run All**, send back these files:

```text
research/ml/artifacts/v4_failure_diagnostics/

V4_FAILURE_DIAGNOSTIC_DECISION.json
V4_FAILURE_TAXONOMY.csv
V4_NEXT_DEV_COLLECTION_PLAN.csv
V4_DIAG_AXIS_BOOTSTRAP.csv                 # if 05 OOF artifact is present
V4_DIAG_AXIS_REFERENCE_DELTA.csv           # if 05 OOF artifact is present
V4_DIAG_ELASTICNET_FEATURE_STABILITY.csv   # if 05 fold-selection artifact is present
```

The primary decision file is:

```text
V4_FAILURE_DIAGNOSTIC_DECISION.json
```

Expected status:

```text
V4_FAIL_DIAGNOSED_COLLECT_UNIFIED_DEV
```

Do **not** open a holdout after this notebook. The next operation is fresh **development** collection under the frozen unified protocol.